# 웹 검색 (Web Search)
웹 검색 기능은 모델이 **인터넷의 최신 정보를 실시간으로 조회**하고, **출처가 명시된 답변(citations 포함)**을 제공할 수 있도록 합니다.
Responses API를 사용하면 API 요청의 tools 배열에서 웹 검색을 구성하여 콘텐츠를 생성함으로써 웹 검색을 활성화할 수 있습니다. 다른 도구와 마찬가지로, 모델은 입력 프롬프트의 내용에 따라 웹 검색 여부를 선택할 수 있습니다.  

### OpenAI 모델이 제공하는 웹 검색의 세 가지 주요 유형

1. **비-추론형 웹 검색 (Non-reasoning web search)**

    * 모델이 사용자의 쿼리를 웹 검색 도구에 직접 전달합니다.
    * 도구는 상위 검색 결과를 기반으로 응답을 반환합니다.
    * 모델 내부에서 별도의 **계획(plan)** 이나 **추론(reasoning)** 은 수행하지 않습니다.
    * **장점:** 매우 빠르고 단순한 조회(lookup)에 적합
    * **예시:** “오늘 서울 날씨” 같은 간단한 질의

2. **에이전트형 검색 (Agentic search with reasoning models)**

    * **추론(reasoning)** 이 가능한 모델이 **검색 과정을 스스로 관리**합니다.
    * 모델은 검색을 체인 오브 소트(chain of thought) 내 일부로 수행하며,
      결과를 분석하고 **추가 검색이 필요한지 여부를 판단**합니다.
    * **특징:** 유연하고 복잡한 워크플로에 적합하지만, 단순 검색보다 시간이 더 걸림
    * **예시:** GPT-5의 reasoning level을 조정하여 검색의 깊이(정밀도)와 속도(지연시간)를 함께 조절 가능

3. **심층 연구 (Deep research)**

    * **에이전트 주도(agent-driven)**의 고급 방식으로,
      **깊이 있고 장시간 진행되는 탐색형 조사**를 수행합니다.
    * 모델은 수백 개의 출처를 활용하며, chain of thought의 일부로 다단계 검색을 실행합니다.
    * **특징:** 몇 분간 실행될 수 있으며, **백그라운드 모드**에서 사용하는 것이 권장됨
    * **주로 사용하는 모델:** `o3-deep-research`, `o4-mini-deep-research`, 또는 **GPT-5 (reasoning level=high)**

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv()) 

True

In [2]:
from openai import OpenAI

client = OpenAI()

Model = "gpt-5-nano"

모델 응답을 생성할 때, 내장 도구와 원격 MCP 서버를 사용하여 기능을 확장할 수 있습니다. 이를 통해 모델이 웹 검색, 파일 검색, 사용자 함수 호출, 또는 외부 서비스 접근이 가능해집니다.

## Web search 

* API 요청 시 `tools` 배열 안에 **web search 도구(web)** 를 추가합니다.
* 모델은 **입력 프롬프트의 내용에 따라**, 필요 시 자동으로 웹 검색을 수행하거나 생략할 수 있습니다.

In [4]:
response = client.responses.create(
    model=Model,
    tools=[{"type": "web_search"}],
    input="오늘 주가 상승에 긍정적인 뉴스는 무엇이었나요??"
)

print(response.output_text)

오늘은 2025년 10월 29일(수) 기준으로 주가 상승에 긍정적 영향을 준 핵심 뉴스는 아래와 같습니다.

- Nvidia의 AI 칩 예약 급증 및 DoE(미 국무부처)용 슈퍼컴퓨터 구축 소식
  - Nvidia가 AI 칩에 대해 5천억 달러 규모의 예약(activity in bookings)을 발표했고, 미국 에너지부를 위한 7대의 슈퍼컴퓨터를 구축하겠다고 밝혔습니다. 이 소식이 시장의 AI 수요 기대감을 크게 끌어올렸습니다. ([reuters.com](https://www.reuters.com/world/china/global-markets-wrapup-1-2025-10-29/))

- Microsoft의 OpenAI 관련 구조 조정 및 지분 소유권 관련 합의
  - Microsoft가 OpenAI를 공익법인(public benefit corporation)으로 재구성하는 데 동의하고, OpenAI의 지분 중 약 27%를 확보하는 역할을 하게 되었어요. 이 소식도 tech/AI 관련주를 강하게 끌어올렸습니다. ([reuters.com](https://www.reuters.com/world/china/global-markets-wrapup-1-2025-10-29/))

- Fed 금리 인하 기대 및 시장 심리
  - 위 뉴스들과 함께 이번 주 연방준비제도(Fed) 회의에서의 금리 인하 기대가 커지며, 채권 수익률과 달러 가치에 변화가 나타나고 있습니다. 시장은 25bp(0.25%)의 금리 인하를 거의 반영하고 있는 분위기이며, 이는 주가에 추가적인 상승 모멘텀으로 작용하고 있습니다. ([reuters.com](https://www.reuters.com/world/china/global-markets-wrapup-1-2025-10-29/))

정리하자면, 오늘의 상승은 AI 기술주에 대한 강한 낙관(In AI)과 대기업의 AI 전략 관련 호재, 그리고 Fed 금리 인하 기대가 맞물린 결과로 보입니다. 필요하시면 오늘의 주요 지수(다우, S&P 500, 나스닥

## 도메인 필터링
웹 검색의 도메인 필터링을 사용하면 검색 결과를 특정 도메인 집합으로 제한할 수 있습니다.  
filters 매개변수를 사용하여 최대 20개의 URL 허용 목록을 설정할 수 있습니다.   
URL 형식을 지정할 때는 HTTP 또는 HTTPS 접두사를 생략하세요. 예를 들어 https://openai.com/ 대신 openai.com을 사용합니다. 이 방식은 하위 도메인도 검색에 포함시킵니다.   
도메인 필터링은 web_search 도구를 사용하는 Responses API에서만 사용할 수 있습니다. 

### Sources
웹 검색 중에 조회된 모든 URL을 보려면 sources 필드를 사용하세요.   
가장 관련성 높은 참조만 보여주는 인라인 인용과 달리, sources는 모델이 응답을 작성할 때 참조한 전체 URL 목록을 반환합니다.   
출처의 수는 종종 인용의 수보다 많습니다. 실시간 제3자 피드도 여기에 표시되며 oai-sports, oai-weather, oai-finance로 라벨이 지정됩니다.   
sources 필드는 web_search 및 web_search_preview 도구 모두에서 사용할 수 있습니다.

In [6]:
response = client.responses.create(
    model=Model,             
    reasoning={"effort": "low"},     # 모델의 추론 강도 설정 (low, medium, high 중 선택)
                                     # 복잡한 논리나 분석이 필요 없는 간단 검색이라 "low"로 지정
    tools=[
        {
            "type": "web_search",    # 사용할 도구: 웹 검색(web_search)     
            "filters": {             # 검색 필터 설정
                "allowed_domains": [ # 특정 도메인만 검색 허용
                    "openai.com",    
                    "github.com",    
                    "stackoverflow.com"  
                ]
            },
        }
    ],
    
    tool_choice="auto",              # 모델이 도구 사용 여부를 자동으로 결정
                                     # (필요시 web_search 실행, 불필요시 내부 지식만 사용)
    include=["web_search_call.action.sources"],  
                                     # 응답 결과에 검색 출처(링크/도메인) 정보를 포함하도록 설정
                                     # 출력 텍스트에는 표시되지 않지만 response 객체에 포함됨
    input="OpenAI의 최신 모델은?",  # 사용자 질문 (입력 프롬프트)
)

# 모델의 텍스트 응답 출력
print(response.output_text)

요약
- OpenAI가 2024년부터 운영해 온 최신 주력 모델군은 GPT-4o 계열입니다. 이 계열에는 GPT-4o와 GPT-4o-mini가 포함됩니다. 또한 2025년 초부터 GPT-4.5 프리뷰 같은 새로운 프리뷰 모델도 등장했습니다. 구체적인 릴리스 날짜를 아래에 적어 두었습니다.

구체 정보와 날짜
- GPT-4o: 2024-11-20 버전이 공개되었다는 기록이 있습니다. ([github.com](https://github.com/openai/simple-evals?utm_source=openai))
- GPT-4o-mini: 2024-07-18 버전이 존재했고, 이후 우선순위가 GPT-4o로 대체되는 흐름이 있었습니다. ([github.com](https://github.com/openai/simple-evals?utm_source=openai))
- GPT-4.5-preview: 2025-02-27 버전으로 프리뷰 모델이 제공되었음을 확인할 수 있습니다. ([github.com](https://github.com/openai/simple-evals?utm_source=openai))

참고
- 최신 모델 구성이 바뀌거나 신규 모델이 추가될 수 있습니다. OpenAI의 공식 발표나 문서를 확인해 두는 것을 권장드립니다. 현재 제시한 날짜는 저장소의 기록에서 확인한 것이며, 실제 서비스에서의 모델 명칭 및 가용성은 계정/지역에 따라 다를 수 있습니다.

필요하시면:
- 현재 본인 계정에서 사용 가능한 모델 목록과 각 모델의 특징(멀티모달 지원 여부, 컨텍스트 길이 등)을 구체적으로 확인하는 방법
- 특정 모델(gpt-4o, gpt-4o-mini, 4.5-preview 등) 간의 차이를 정리한 비교표
를 더 자세히 정리해 드리겠습니다.


### 사용자 위치  
지역을 기준으로 검색 결과를 좁히려면 국가, 도시, 지역 및/또는 시간대를 사용하여 대략적인 사용자 위치를 지정할 수 있습니다.  
도시 및 지역 필드는 각각 미니애폴리스와 미네소타처럼 자유 텍스트 문자열입니다.  
국가 필드는 US처럼 두 글자 ISO 국가 코드입니다.

In [7]:
response = client.responses.create(
    model=Model,
    tools=[{
        "type": "web_search",
        "user_location": {
            "type": "approximate",
            "country": "KR",
            "city": "서울",
            "region": "서울",
        }
    }],
    input="종로 주변 최고의 레스토랑은 어디인가요??",
)

print(response.output_text)

좋은 선택이에요! 종로 주변은 전통 한식부터 현대 한식까지 다양한 옵션이 있어 취향에 따라 골라 보실 수 있는데요, 2025-10-29 기준으로 현장 분위기와 미쉐린 가이드의 권고를 종합해 대표적으로 많이 손꼽히는 곳들을 정리해 드릴게요.

추천 포인트별 정리
- Onjium (온지엄) – 미쉐린 1스타, 종로구 효자로
  - 특징: 한국 왕실 요리를 현대적으로 재해석한 고급 코스 요리. 카운터 좌석에서 셰프의 진행을 바로 볼 수 있는 체험형 다이닝이고, 정교한 tasting menu를 즐길 수 있습니다.
  - 분위기/가격: 매우 고급스러운 분위기, 다소 비싼 편.
  - 위치: 4F, 49 Hyoja-ro, Jongno-gu
  - 근거: 미쉐린 가이드 페이지에서 하나의 스타 등재 및 위치 정보 확인 가능. ([guide.michelin.com](https://guide.michelin.com/kr/en/seoul-capital-area/kr-seoul/restaurant/onjium?utm_source=openai))

- Restaurant Jueun (레스토랑 주은) – 미쉐린 가이드에 등재된 한식 고급 레스토랑
  - 특징: 전통 한식 요리를 현대적 미감으로 선보이는 레스토랑. 고급스러운 공간 구성과 함께 한식 코스가 주력.
  - 분위기/가격: 고급 레스토랑 수준의 가격대 ₩₩₩.
  - 위치: 8F, Gyeonghuidang, 36 Gyeonghuigung-gil, Jongno-gu
  - 근거: 미쉐린 가이드 페이지에 레스토랑 정보 및 위치 제공. ([guide.michelin.com](https://guide.michelin.com/kr/en/seoul-capital-area/kr-seoul/restaurant/restaurant-jueun))

- Niroumianguan (니로우미앙관) – Bib Gourmand(가성비 좋은 맛집 카테고리), 면 요리 중심
  - 특징: 중국 북경식 소고기 면 요리(라면 스타일)가 대표 메뉴인 소박하지만 